<a href="https://colab.research.google.com/github/SKaiForge/amt/blob/main/amt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [67]:
!nvidia-smi

Mon Nov 16 13:56:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             27W /   70W |   11953MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [68]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [69]:
import shutil
from pathlib import Path

cache_source = Path("/content/drive/MyDrive/huggingface-cache")
cache_target = Path("/content/huggingface-cache")
if cache_source.exists() and not cache_target.exists():
    shutil.copytree(cache_source, cache_target)
cache_target.mkdir(parents=True, exist_ok=True)
print("Local model cache:", cache_target)


Local model cache: /content/huggingface-cache


In [70]:
# !cp -r /content/huggingface-cache /content/drive/MyDrive/huggingface-cache

In [71]:
import os

persistent_cache = "/content/huggingface-cache"
os.environ["HF_HOME"] = persistent_cache
os.environ["HUGGINGFACE_HUB_CACHE"] = persistent_cache
# Online fallback is allowed by default if a snapshot is missing.
# Set AMT_OFFLINE=1 before this cell for a fully cached, offline run.
offline = os.environ.get("AMT_OFFLINE", "0")
os.environ["HF_HUB_OFFLINE"] = offline
os.environ["TRANSFORMERS_OFFLINE"] = offline


In [72]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3)

2.14.1+cu130
True
Tesla T4
14.56317138671875


In [73]:
from pathlib import Path
from google.colab import userdata
import subprocess

repo_path = Path("/content/amt")
if (repo_path / ".git").exists():
    subprocess.run(["git", "-C", str(repo_path), "pull", "--ff-only"], check=True)
else:
    token = userdata.get("GITHUB_TOKEN")
    netrc = Path.home() / ".netrc"
    previous_netrc = netrc.read_bytes() if netrc.exists() else None
    previous_mode = netrc.stat().st_mode & 0o777 if netrc.exists() else None
    try:
        netrc.write_text(
            f"machine github.com login x-access-token password {token}\n", encoding="utf-8",
        )
        netrc.chmod(0o600)
        subprocess.run(["git", "clone", "https://github.com/SKaiForge/amt.git", str(repo_path)], check=True)
    finally:
        if previous_netrc is None:
            netrc.unlink(missing_ok=True)
        else:
            netrc.write_bytes(previous_netrc)
            netrc.chmod(previous_mode)
        del token
print("Repository ready:", repo_path)


Repository ready: /content/amt


In [74]:
import sys

repo_src = "/content/amt/src"
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

In [75]:
%pip install transformers==4.57.1 accelerate==1.11.0 safetensors==0.6.2 sentencepiece==0.2.1 huggingface-hub==0.36.0 bitsandbytes==0.48.2


In [76]:
# Dependencies are pinned together in the preceding cell.
# Start from a fresh Colab runtime when changing installed versions.


In [77]:
import torch
import transformers
import bitsandbytes

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Torch: 2.14.1+cu130
CUDA available: True
Transformers: 4.57.1
GPU: Tesla T4


In [78]:
import os
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

cache_dir = "/content/huggingface-cache"
model_id = "Qwen/Qwen3-8B"
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before loading the model.")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Set AMT_MODEL_REVISION to reuse a specific model commit across sessions.
requested_revision = os.environ.get("AMT_MODEL_REVISION", "main")
config = AutoConfig.from_pretrained(model_id, revision=requested_revision, cache_dir=cache_dir)
MODEL_REVISION = getattr(config, "_commit_hash", None)
if not MODEL_REVISION:
    raise RuntimeError("Could not resolve an immutable model commit.")
print("Pinned model/tokenizer revision:", MODEL_REVISION)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype,
)
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=MODEL_REVISION, cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=MODEL_REVISION, config=config,
    quantization_config=quantization_config, device_map="auto",
    cache_dir=cache_dir, torch_dtype=dtype,
)
model.requires_grad_(False)
model.eval()


Pinned model/tokenizer revision: b968826d9c46dd6066d109eabc6255188de91218


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
messages = [
    {
        "role": "system",
        "content": "Answer using the supplied user memory.",
    },
    {
        "role": "user",
        "content": (
            "Memory:\n"
            "The user prefers Python for ML experiments.\n\n"
            "Question: What language should the example use?"
        ),
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
    )

generated = outputs[0, inputs.input_ids.shape[1]:]
answer = tokenizer.decode(generated, skip_special_tokens=True)

print(answer)

In [ ]:
# %cd /content/amt

# !PYTHONPATH=/content/amt/src python3 -m datasets.amt_memorybench amt-100

In [ ]:
# %cd /content/amt

# !PYTHONPATH=/content/amt/src python3 -m datasets.amt_memorybench smoke

In [ ]:
# %cd /content/amt

# !PYTHONPATH=/content/amt/src python3 -m datasets.amt_memorybench atomic-smoke

In [ ]:
%cd /content/amt

!PYTHONPATH=src python3 -m datasets.amt_memorybench atomic-smoke-v2

In [ ]:
# from pathlib import Path
# from datasets.amt_memorybench.answer import TextMemoryAnswerer
# from datasets.amt_memorybench.evaluate import (
#     LexicalMemorySelector,
#     read_jsonl,
# )
# from datasets.amt_memorybench.answer import evaluate_answers


# class QwenTextModel:
#     def __init__(self, model, tokenizer):
#         self.model = model
#         self.tokenizer = tokenizer

#     def generate(self, prompt: str) -> str:
#         messages = [{"role": "user", "content": prompt}]

#         text = self.tokenizer.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=True,
#             enable_thinking=False,
#         )
#         print(f"prompt: {prompt}")
#         inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)

#         with torch.inference_mode():
#             outputs = self.model.generate(
#                 **inputs,
#                 max_new_tokens=64,
#                 do_sample=False,
#             )

#         generated = outputs[0, inputs.input_ids.shape[1]:]
#         answer = self.tokenizer.decode(generated, skip_special_tokens=True)
#         print(f"answer: {answer}")
#         return answer


# qwen_model = QwenTextModel(model, tokenizer)
# answerer = TextMemoryAnswerer(qwen_model)

# # Wrapped the string path with Path() and pointed it to the cloned repository folder
# examples = read_jsonl(
#     Path("/content/amt/data/processed/amt_memorybench/amt-smoke.jsonl")
# )

# metrics = evaluate_answers(
#     examples,
#     LexicalMemorySelector(),
#     answerer,
#     k=1,
# )

# print(metrics)

In [ ]:
import importlib.metadata
import json
import platform
import subprocess
from pathlib import Path

TOKEN_COUNTS = (1, 2, 4)
SEEDS = (17, 23, 42)
TRAIN_STEPS = 100
LEARNING_RATE = 1e-2
MAX_NEW_TOKENS = 16
MAX_SESSION_HOURS = 6
REQUIRE_TEXT_GATE = True
TEXT_GATE_ALIAS_ACCURACY = 0.95
RUN_ROOT = Path("/content/drive/MyDrive/amt-experiments/atomic-smoke-v2-capacity")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

model.requires_grad_(False)
model.zero_grad(set_to_none=True)
model.eval()
assert not any(parameter.requires_grad for parameter in model.parameters())
print("Free-vector diagnostic only: no shared projector is allocated.")
print("Capacity:", TOKEN_COUNTS, "Seeds:", SEEDS, "Steps:", TRAIN_STEPS)


In [ ]:
# examples = read_jsonl(
#     Path("/content/amt/data/processed/amt_memorybench/amt-smoke.jsonl")
# )

# single_recall = [
#     example
#     for example in examples
#     if example.task_type == "single_recall"
# ]

# def gold_postulates(example):
#     required_ids = set(example.required_memory_ids)
#     return [
#         memory.text
#         for memory in example.memories
#         if memory.memory_id in required_ids
#     ]

# def amt_prompt(question):
#     return (
#         "Answer with only the shortest answer span. "
#         "Do not explain or add advice.\n\n"
#         f"Question: {question}\nAnswer:"
#     )

In [ ]:
# train_examples = [
#     example for example in single_recall
#     if example.example_id.endswith("_00")
# ]

# test_examples = [
#     example for example in single_recall
#     if example.example_id.endswith("_01")
# ]

# len(train_examples), len(test_examples)

In [ ]:
# import torch
# from pathlib import Path
# from datasets.amt_memorybench.evaluate import read_jsonl

# # 1. Reset any monkey-patched forward methods on the base model
# if 'model' in globals():
#     try:
#         del model.forward
#     except AttributeError:
#         pass

# # 2. Re-initialize the AMT wrapper to completely wipe the existing NaN-corrupted parameters
# if 'amt' in globals():
#     del amt

# from amt_memory.models import QwenPrefixAMT
# amt = QwenPrefixAMT(model, tokenizer)

# # 3. Load dataset and prepare training/testing splits
# examples = read_jsonl(
#     Path("/content/amt/data/processed/amt_memorybench/amt-smoke.jsonl")
# )

# single_recall = [
#     example
#     for example in examples
#     if example.task_type == "single_recall"
# ]

# def gold_postulates(example):
#     required_ids = set(example.required_memory_ids)
#     return [
#         memory.text
#         for memory in example.memories
#         if memory.memory_id in required_ids
#     ]

# def amt_prompt(question):
#     return (
#         "Answer with only the shortest answer span. "
#         "Do not explain or add advice.\n\n"
#         f"Question: {question}\nAnswer:"
#     )

# train_examples = [
#     example for example in single_recall
#     if example.example_id.endswith("_00")
# ]

# test_examples = [
#     example for example in single_recall
#     if example.example_id.endswith("_01")
# ]

# # Ensure the base model vocabulary embeddings remain completely frozen
# if hasattr(amt, 'embedding'):
#     for param in amt.embedding.parameters():
#         param.requires_grad = False

# # 4. Extract ONLY the actual trainable projector module
# trainable_modules = []
# for attr_name in dir(amt):
#     attr = getattr(amt, attr_name)
#     if isinstance(attr, torch.nn.Module) and attr_name not in ["model", "embedding"]:
#         trainable_modules.append(attr)

# # Force the projector layers to be stored in highly stable FP32
# for module in trainable_modules:
#     module.to(dtype=torch.float32)

# # Intercept projector forward pass to safely run calculation in FP32
# original_projector_forward = amt.projector.forward

# def safe_projector_forward(x):
#     # 1. Upcast input activations to float32
#     x_f32 = x.to(torch.float32)
#     # 2. Compute projections in stable float32
#     out_f32 = original_projector_forward(x_f32)
#     # 3. Downcast back to the base model's precision (bfloat16 or float16)
#     return out_f32.to(dtype)

# amt.projector.forward = safe_projector_forward

# trainable_params = []
# for module in trainable_modules:
#     for param in module.parameters():
#         if param.requires_grad:
#             # Clean re-initialization of weights to wipe out the NaN pollution
#             torch.nn.init.normal_(param, mean=0.0, std=0.02)
#             trainable_params.append(param)

# # 5. Setup optimizer with standard learning rate for training the FP32 projector
# optimizer = torch.optim.AdamW(
#     trainable_params,
#     lr=1e-4,
#     weight_decay=0.01,
#     eps=1e-8
# )

# # 6. Training loop
# for module in trainable_modules:
#     module.train()

# for epoch in range(30):
#     total_loss = 0.0
#     successful_steps = 0

#     for example in train_examples:
#         optimizer.zero_grad(set_to_none=True)

#         loss = amt.loss(
#             postulates=gold_postulates(example),
#             prompt=amt_prompt(example.query),
#             answer=example.answer,
#         )

#         if torch.isnan(loss) or torch.isinf(loss):
#             continue

#         loss.backward()
#         # Clip gradients to keep updates extremely stable
#         torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=0.5)
#         optimizer.step()

#         total_loss += loss.item()
#         successful_steps += 1

#     if epoch % 5 == 0:
#         avg_loss = total_loss / successful_steps if successful_steps > 0 else float('nan')
#         print(f"epoch={epoch:02d} loss={avg_loss:.4f}")

In [ ]:
# import re

# def normalize(text):
#     # Lowercase, replace non-alphanumeric characters with spaces, and split into words
#     cleaned = re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower())
#     return cleaned.split()

# correct = 0

# amt.eval()

# for example in test_examples:
#     prediction = amt.generate(
#         postulates=gold_postulates(example),
#         query=amt_prompt(example.query),
#         max_new_tokens=16,
#     )

#     # Get normalized text strings
#     expected_words = normalize(example.answer)
#     actual_words = normalize(prediction)

#     expected_str = " ".join(expected_words)
#     actual_str = " ".join(actual_words)

#     # Allow partial match (e.g., if gold is 'PyTorch for ML' and prediction contains 'pytorch')
#     # checking if key terms in expected are found in the actual prediction
#     is_correct = any(word in actual_words for word in expected_words if word not in ['for', 'ml', 'experiments'])
#     correct += is_correct

#     print("Question:", example.query)
#     print("Gold:", example.answer)
#     print("Prediction:", prediction.strip())
#     print("Correct:", bool(is_correct))
#     print()

# print("H1/H3 held-out-paraphrase accuracy:", correct / len(test_examples))

In [ ]:
# import torch
# import torch.nn as nn

# print("=== Starting Diagnostics & Controls ===")

# # ---------------------------------------------------------
# # Control 1: Text-Memory Baseline Control
# # ---------------------------------------------------------
# print("\n--- Running Control: Text-Memory Baseline ---")
# prompt_text = (
#     "Answer with only the shortest answer span.\n\n"
#     "Memory: The user prefers Julia for ML experiments.\n"
#     "Question: Which framework is preferred?\n"
#     "Answer:"
# )

# inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
# with torch.inference_mode():
#     outputs = model.generate(**inputs, max_new_tokens=16, do_sample=False)
# text_generation = tokenizer.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
# print(f"Text Control Generation: '{text_generation.strip()}'")

# # ---------------------------------------------------------
# # Experiment: Multi-Token Free-Vector Overfit (100 steps)
# # ---------------------------------------------------------
# print("\n--- Running Experiment: multiple-Token Free-Vector Overfit ---")
# token_count = 1
# hidden_size = model.config.hidden_size
# dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# # Using 5 prefix tokens to guarantee capacity
# free_prefix = nn.Parameter(torch.randn(1, token_count, hidden_size, device=model.device, dtype=dtype) * 0.02)

# def amt_prompt(question):
#     return (
#         "Answer with only the shortest answer span. "
#         "Do not explain or add advice.\n\n"
#         f"Question: {question}\nAnswer:"
#     )

# question_prompt = amt_prompt("Which framework is preferred?")
# target_answer = " Julia"

# prompt_ids = tokenizer(question_prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
# answer_ids = tokenizer(target_answer, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
# eos = torch.tensor(
#     [[tokenizer.eos_token_id]],
#     dtype=torch.long,
#     device=model.device,
# )
# answer_ids = torch.cat([answer_ids, eos], dim=1)

# optimizer = torch.optim.AdamW([free_prefix], lr=1e-2)  # slightly higher LR for faster convergence

# for step in range(101):
#     optimizer.zero_grad(set_to_none=True)

#     with torch.no_grad():
#         prompt_embeds = model.get_input_embeddings()(prompt_ids)
#         answer_embeds = model.get_input_embeddings()(answer_ids)

#     prefix_for_model = free_prefix.to(prompt_embeds.dtype)

#     full_embeds = torch.cat(
#         [prefix_for_model, prompt_embeds, answer_embeds],
#         dim=1,
#     )

#     # Ignore prefix and prompt; supervise every answer token.
#     full_labels = torch.full(
#         full_embeds.shape[:2],
#         -100,
#         dtype=torch.long,
#         device=model.device,
#     )
#     answer_start = token_count + prompt_ids.shape[1]

#     full_labels[:, answer_start:] = answer_ids

#     outputs = model(
#         inputs_embeds=full_embeds,
#         attention_mask=torch.ones(
#             full_embeds.shape[:2],
#             dtype=torch.long,
#             device=model.device,
#         ),
#         labels=full_labels,
#     )

#     loss = outputs.loss
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_([free_prefix], 1.0)
#     optimizer.step()

#     if step % 10 == 0:
#         print(step, loss.item(), free_prefix.grad.norm().item())

In [ ]:
# prompt_embeds = model.get_input_embeddings()(prompt_ids)
# inputs_embeds = torch.cat([free_prefix, prompt_embeds], dim=1)

# attention_mask = torch.ones(
#     inputs_embeds.shape[:2],
#     dtype=torch.long,
#     device=model.device,
# )

# with torch.inference_mode():
#     generated = model.generate(
#         inputs_embeds=inputs_embeds,
#         attention_mask=attention_mask,
#         max_new_tokens=8,
#         do_sample=False,
#     )

# print(tokenizer.decode(generated[0], skip_special_tokens=True))

In [ ]:
# test_prompt = amt_prompt("What ML framework does the user prefer?")
# test_ids = tokenizer(
#     test_prompt,
#     return_tensors="pt",
# ).input_ids.to(model.device)

# test_embeds = model.get_input_embeddings()(test_ids)
# test_inputs = torch.cat([free_prefix, test_embeds], dim=1)

# with torch.inference_mode():
#     generated = model.generate(
#         inputs_embeds=test_inputs,
#         attention_mask=torch.ones(
#             test_inputs.shape[:2],
#             dtype=torch.long,
#             device=model.device,
#         ),
#         max_new_tokens=8,
#         do_sample=False,
#     )

# print(tokenizer.decode(generated[0], skip_special_tokens=True))

In [ ]:
# import torch

# embedding_layer = model.get_input_embeddings()
# model.eval()

# def amt_prompt(question: str) -> str:
#     return (
#         "Answer with only the shortest answer span. "
#         "Do not explain or add advice.\n\n"
#         f"Question: {question}\nAnswer:"
#     )


# def generate_from_prefix(prefix: torch.Tensor, question: str) -> str:
#     prompt = amt_prompt(question)

#     prompt_ids = tokenizer(
#         prompt,
#         return_tensors="pt",
#         add_special_tokens=True,
#     ).input_ids.to(model.device)

#     with torch.no_grad():
#         prompt_embeds = embedding_layer(prompt_ids)

#     prefix = prefix.to(
#         device=prompt_embeds.device,
#         dtype=prompt_embeds.dtype,
#     )

#     inputs_embeds = torch.cat(
#         [prefix, prompt_embeds],
#         dim=1,
#     )

#     attention_mask = torch.ones(
#         inputs_embeds.shape[:2],
#         dtype=torch.long,
#         device=model.device,
#     )

#     with torch.inference_mode():
#         generated = model.generate(
#             inputs_embeds=inputs_embeds,
#             attention_mask=attention_mask,
#             max_new_tokens=8,
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id,
#         )

#     return tokenizer.decode(
#         generated[0],
#         skip_special_tokens=True,
#     ).strip()

In [ ]:
# original_question = "Which framework is preferred?"
# paraphrase_question = "What ML framework does the user prefer?"

# print("=== Correct learned prefix ===")
# print("Original:", generate_from_prefix(free_prefix.detach(), original_question))
# print("Paraphrase:", generate_from_prefix(free_prefix.detach(), paraphrase_question))

In [ ]:
# zero_prefix = torch.zeros_like(free_prefix)

# random_generator = torch.Generator(device=model.device)
# random_generator.manual_seed(17)

# random_prefix = torch.randn(
#     free_prefix.shape,
#     generator=random_generator,
#     device=free_prefix.device,
#     dtype=free_prefix.dtype,
# ) * 0.02

# print("\n=== Zero prefix ===")
# print(generate_from_prefix(zero_prefix, original_question))

# print("\n=== Random prefix ===")
# print(generate_from_prefix(random_prefix, original_question))

In [ ]:
import gc
import hashlib
from collections import defaultdict
from pathlib import Path

import torch
import torch.nn as nn
from datasets.amt_memorybench.evaluate import read_jsonl

benchmark_path = Path(
    "/content/amt/data/processed/amt_memorybench/amt-atomic-smoke-v2.jsonl"
)
examples = read_jsonl(benchmark_path)
groups = defaultdict(list)
for example in examples:
    if not example.paraphrase_group_id:
        raise ValueError(f"Missing paraphrase group: {example.example_id}")
    groups[example.paraphrase_group_id].append(example)

unique_pairs = {}
for group_id, group in groups.items():
    pair = sorted(group, key=lambda example: example.example_id)
    if len(pair) != 2:
        raise ValueError(f"Expected two examples in {group_id}")
    train_example, test_example = pair
    train_memories = tuple(
        memory for memory in train_example.memories
        if memory.memory_id in train_example.required_memory_ids
    )
    test_memories = tuple(
        memory for memory in test_example.memories
        if memory.memory_id in test_example.required_memory_ids
    )
    if (len(train_memories) != 1 or train_memories != test_memories
            or train_example.answer != test_example.answer
            or train_example.query == test_example.query):
        raise ValueError(f"Invalid atomic paraphrase pair: {group_id}")
    memory = train_memories[0]
    key = (memory.type, memory.text, train_example.answer)
    unique_pairs.setdefault(key, (train_example, test_example))

selected_pairs = []
for memory_type in sorted({key[0] for key in unique_pairs}):
    candidates = sorted(
        (pair for key, pair in unique_pairs.items() if key[0] == memory_type),
        key=lambda pair: (pair[0].answer, pair[0].example_id),
    )
    distinct_answers = {}
    for pair in candidates:
        distinct_answers.setdefault(pair[0].answer, pair)
    if len(distinct_answers) < 2:
        raise ValueError(f"Need two different values for {memory_type}")
    selected_pairs.extend(list(distinct_answers.values())[:2])

assert len(selected_pairs) == 12
train_examples = [pair[0] for pair in selected_pairs]
test_examples = [pair[1] for pair in selected_pairs]
print(f"Loaded {len(groups)} groups; {len(unique_pairs)} distinct postulates")
print(f"Selected {len(selected_pairs)} pairs, two different values per type")
for train_example, test_example in selected_pairs:
    print(train_example.example_id, train_example.answer, "->", test_example.query)


In [ ]:
import hashlib
import marshal
import random
import time
import unicodedata

embedding_layer = model.get_input_embeddings()
hidden_size = model.config.hidden_size
input_device = embedding_layer.weight.device
if input_device.type != "cuda":
    raise RuntimeError("This diagnostic requires CUDA input embeddings.")
if any(str(device) in {"cpu", "disk"} for device in getattr(model, "hf_device_map", {}).values()):
    raise RuntimeError("CPU/disk offload detected; use a GPU-fitting model for input-gradient training.")

PROMPT_STYLE = "qwen3-chat-nonthinking-short-span"
ANSWER_STOP_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
if ANSWER_STOP_ID is None or tokenizer.convert_ids_to_tokens(ANSWER_STOP_ID) != "<|im_end|>":
    raise ValueError("A Qwen chat end-of-turn token is required.")
APPROVED_ALIASES = {
    "16 GB of memory": ("fit on 16 GB of memory",),
    "a 30-minute runtime": ("30-minute runtime",),
    "a Python meetup": ("Python meetup",),
    "a data engineering summit": ("data engineering summit",),
    "a machine-learning workshop": (
        "a machine learning workshop", "the machine learning workshop",
        "the machine-learning workshop", "machine learning workshop",
    ),
}


def prompt_for(question, memory_text=None):
    memory = f"Memory: {memory_text}\n\n" if memory_text is not None else ""
    messages = [
        {"role": "system", "content": (
            "Answer using the supplied user memory. Return only the shortest exact answer span. "
            "Preserve the memory's wording, articles, hyphens, numbers, and units. "
            "Do not add explanations, labels, or punctuation. "
            "If the answer is not available in memory, return unknown."
        )},
        {"role": "user", "content": f"{memory}Question: {question}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )


def normalize(text):
    return " ".join(unicodedata.normalize("NFC", text).casefold().split())


def score_answer(prediction, gold):
    strict = normalize(prediction) == normalize(gold)
    aliases = {normalize(value) for value in APPROVED_ALIASES.get(gold, ())}
    return {"strict": strict, "alias": strict or normalize(prediction) in aliases}


def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_answer_ids(answer):
    answer_ids = tokenizer(
        answer, return_tensors="pt", add_special_tokens=False,
    ).input_ids.to(input_device)
    end_of_turn = torch.tensor([[ANSWER_STOP_ID]], dtype=torch.long, device=input_device)
    return torch.cat([answer_ids, end_of_turn], dim=1)


@torch.inference_mode()
def generate_from_prefix(prefix, question, max_new_tokens=MAX_NEW_TOKENS, memory_text=None):
    prompt_ids = tokenizer(
        prompt_for(question, memory_text), return_tensors="pt", add_special_tokens=False,
    ).input_ids.to(input_device)
    prompt_embeds = embedding_layer(prompt_ids)
    if prefix is None:
        prefix = torch.empty((1, 0, hidden_size), device=input_device)
    inputs_embeds = torch.cat([
        prefix.to(device=input_device, dtype=prompt_embeds.dtype), prompt_embeds,
    ], dim=1)
    generated = model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=input_device),
        max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
        pad_token_id=ANSWER_STOP_ID, eos_token_id=ANSWER_STOP_ID,
    )
    return tokenizer.decode(generated[0], skip_special_tokens=True).strip()


def train_one_free_token(example, token_count, seed, steps=TRAIN_STEPS, learning_rate=LEARNING_RATE):
    seed_everything(seed)
    free_prefix = nn.Parameter(torch.randn(
        1, token_count, hidden_size, device=input_device, dtype=torch.float32,
    ) * 0.02)
    optimizer = torch.optim.AdamW([free_prefix], lr=learning_rate, eps=1e-6, weight_decay=0.0)
    prompt_ids = tokenizer(
        prompt_for(example.query), return_tensors="pt", add_special_tokens=False,
    ).input_ids.to(input_device)
    answer_ids = build_answer_ids(example.answer)
    with torch.no_grad():
        prompt_embeds = embedding_layer(prompt_ids).detach()
        answer_embeds = embedding_layer(answer_ids).detach()
    answer_start = token_count + prompt_ids.shape[1]
    labels = torch.full(
        (1, answer_start + answer_ids.shape[1]), -100, dtype=torch.long, device=input_device,
    )
    labels[:, answer_start:] = answer_ids
    mask = torch.ones_like(labels)
    losses, trace = [], []

    def forward_loss():
        full_embeds = torch.cat([
            free_prefix.to(prompt_embeds.dtype), prompt_embeds, answer_embeds,
        ], dim=1)
        return model(inputs_embeds=full_embeds, attention_mask=mask, labels=labels, use_cache=False)

    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)
        outputs = forward_loss()
        loss = outputs.loss
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss at step {step}")
        if step == 0:
            manual_loss = torch.nn.functional.cross_entropy(
                outputs.logits[:, :-1, :].float().reshape(-1, outputs.logits.shape[-1]),
                labels[:, 1:].reshape(-1), ignore_index=-100,
            )
            if not torch.allclose(loss.float(), manual_loss, rtol=0.02, atol=0.02):
                raise RuntimeError("Built-in causal loss differs from manual shifted loss.")
            del manual_loss
        loss.backward()
        if free_prefix.grad is None or not torch.isfinite(free_prefix.grad).all():
            raise FloatingPointError(f"Missing/non-finite gradient at step {step}")
        grad_norm = float(free_prefix.grad.norm())
        torch.nn.utils.clip_grad_norm_([free_prefix], 1.0)
        optimizer.step()
        if not torch.isfinite(free_prefix).all():
            raise FloatingPointError(f"Non-finite prefix after step {step}")
        losses.append(loss.item())
        if step % 10 == 0 or step == steps - 1:
            item = {"step": step, "loss": loss.item(), "grad_norm": grad_norm,
                    "prefix_norm": float(free_prefix.detach().norm())}
            trace.append(item)
            print(item, flush=True)
        del outputs, loss

    with torch.inference_mode():
        final_outputs = forward_loss()
        per_token = torch.nn.functional.cross_entropy(
            final_outputs.logits[:, answer_start - 1:-1, :].float().reshape(-1, final_outputs.logits.shape[-1]),
            answer_ids.reshape(-1), reduction="none",
        ).cpu().tolist()
        final_loss = final_outputs.loss.item()
    return {
        "prefix": free_prefix.detach().cpu(), "losses": losses, "trace": trace,
        "final_loss": final_loss, "answer_token_ids": answer_ids.cpu().tolist()[0],
        "answer_token_losses": per_token[:-1], "eos_loss": per_token[-1],
        "post_training_token_loss": sum(per_token) / len(per_token),
    }


def atomic_json(path, value):
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def memory_for(example):
    return next(memory for memory in example.memories if memory.memory_id in example.required_memory_ids)


def code_fingerprint():
    functions = [prompt_for, normalize, score_answer, seed_everything, build_answer_ids,
                 generate_from_prefix.__wrapped__, train_one_free_token, atomic_json, memory_for]
    return hashlib.sha256(b"".join(marshal.dumps(function.__code__) for function in functions)).hexdigest()


In [ ]:
import types


def stable_helper_fingerprint():
    def canonical_code(code):
        constants = tuple(canonical_code(value) if isinstance(value, types.CodeType) else value
                          for value in code.co_consts)
        return code.replace(co_filename="<amt-capacity>", co_firstlineno=1, co_consts=constants)

    functions = [prompt_for, normalize, score_answer, seed_everything, build_answer_ids,
                 generate_from_prefix.__wrapped__, train_one_free_token, atomic_json, memory_for]
    return hashlib.sha256(b"".join(marshal.dumps(canonical_code(function.__code__))
                                   for function in functions)).hexdigest()


# IPython execution labels must not change the resume fingerprint.
code_fingerprint = stable_helper_fingerprint


In [ ]:
import time

session_started = time.perf_counter()
code_commit = subprocess.check_output(["git", "-C", "/content/amt", "rev-parse", "HEAD"], text=True).strip()
code_diff = subprocess.check_output(["git", "-C", "/content/amt", "diff", "HEAD"], text=True)
model_revision = getattr(model.config, "_commit_hash", None)
if not model_revision:
    raise RuntimeError("Model revision was not resolved; reload from a pinned revision.")
manifest = {
    "experiment": "free-prefix-capacity-v2", "runner_version": 3,
    "code_commit": code_commit, "dirty_diff_sha256": hashlib.sha256(code_diff.encode()).hexdigest(),
    "helper_sha256": code_fingerprint(),
    "dataset_sha256": hashlib.sha256(benchmark_path.read_bytes()).hexdigest(),
    "model_id": model_id, "model_revision": model_revision,
    "tokenizer_revision": MODEL_REVISION, "prompt_style": PROMPT_STYLE,
    "enable_thinking": False, "answer_stop_id": ANSWER_STOP_ID,
    "quantization": quantization_config.to_dict(), "compute_dtype": str(dtype),
    "token_counts": list(TOKEN_COUNTS), "seeds": list(SEEDS), "steps": TRAIN_STEPS,
    "learning_rate": LEARNING_RATE, "max_new_tokens": MAX_NEW_TOKENS,
    "optimizer": {"name": "AdamW", "eps": 1e-6, "weight_decay": 0.0},
    "aliases": APPROVED_ALIASES, "require_text_gate": REQUIRE_TEXT_GATE,
    "text_gate": TEXT_GATE_ALIAS_ACCURACY,
    "python": platform.python_version(), "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "packages": {name: importlib.metadata.version(name) for name in (
        "torch", "transformers", "accelerate", "bitsandbytes", "safetensors", "sentencepiece", "huggingface-hub",
    )},
    "pairs": [{"train_id": train.example_id, "test_id": test.example_id,
               "postulate": memory_for(train).text, "answer": train.answer}
              for train, test in selected_pairs],
}
run_hash = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
run_dir = RUN_ROOT / run_hash[:16]
run_dir.mkdir(parents=True, exist_ok=True)
manifest_path = run_dir / "manifest.json"
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != json.loads(json.dumps(manifest)):
        raise RuntimeError("Run manifest mismatch; refusing resume.")
else:
    atomic_json(manifest_path, manifest)
    (run_dir / "code.diff").write_text(code_diff, encoding="utf-8")
print("Run directory:", run_dir, flush=True)


def check_session_budget():
    if time.perf_counter() - session_started > MAX_SESSION_HOURS * 3600:
        raise TimeoutError("Session budget reached. Completed checkpoints are safe; rerun to resume.")


# E0: Save every prediction before attempting any expensive prefix training.
baseline_path = run_dir / "baselines.json"
baselines = json.loads(baseline_path.read_text()) if baseline_path.exists() else []
baseline_keys = {(item["example_id"], item["condition"]) for item in baselines}
for train_example, test_example in selected_pairs:
    for example in (train_example, test_example):
        for condition in ("text_memory", "no_memory"):
            key = (example.example_id, condition)
            if key in baseline_keys:
                continue
            check_session_budget()
            started = time.perf_counter()
            prediction = generate_from_prefix(
                None, example.query,
                memory_text=memory_for(example).text if condition == "text_memory" else None,
            )
            baselines.append({
                "example_id": example.example_id, "condition": condition,
                "gold": example.answer, "prediction": prediction,
                "scores": score_answer(prediction, example.answer),
                "seconds": time.perf_counter() - started,
            })
            atomic_json(baseline_path, baselines)
            baseline_keys.add(key)
            print(condition, example.example_id, repr(prediction), flush=True)
text_items = [item for item in baselines if item["condition"] == "text_memory"]
text_accuracy = sum(item["scores"]["alias"] for item in text_items) / len(text_items)
print("Text-memory alias accuracy:", text_accuracy, flush=True)
if REQUIRE_TEXT_GATE and text_accuracy < TEXT_GATE_ALIAS_ACCURACY:
    raise RuntimeError("Text-memory gate failed. Inspect baselines.json before changing prompts or starting training.")

# One checkpoint per completed fit. Interrupted fits restart; completed fits are reused.
for token_count in TOKEN_COUNTS:
    for seed in SEEDS:
        for index, (train_example, test_example) in enumerate(selected_pairs):
            checkpoint_path = run_dir / f"prefix-k{token_count}-seed{seed}-pair{index:02d}.pt"
            if checkpoint_path.exists():
                continue
            check_session_budget()
            print(f"Training k={token_count} seed={seed} pair={index + 1}/{len(selected_pairs)}", flush=True)
            torch.cuda.reset_peak_memory_stats()
            started = time.perf_counter()
            fit = train_one_free_token(train_example, token_count=token_count, seed=seed)
            fit["train_seconds"] = time.perf_counter() - started
            fit["peak_gpu_bytes"] = torch.cuda.max_memory_allocated()
            payload = {"run_hash": run_hash, "train_id": train_example.example_id,
                       "token_count": token_count, "seed": seed, "fit": fit}
            temporary = checkpoint_path.with_suffix(".pt.part")
            torch.save(payload, temporary)
            temporary.replace(checkpoint_path)
            print("Saved", checkpoint_path.name, flush=True)
            del fit, payload
            gc.collect()

# Evaluate only after both same-type partners have checkpoints.
result_path = run_dir / "results.json"
results = json.loads(result_path.read_text()) if result_path.exists() else []
result_keys = {(item["token_count"], item["seed"], item["index"]) for item in results}
for token_count in TOKEN_COUNTS:
    for seed in SEEDS:
        for index, (train_example, test_example) in enumerate(selected_pairs):
            key = (token_count, seed, index)
            if key in result_keys:
                continue
            check_session_budget()
            checkpoint_path = run_dir / f"prefix-k{token_count}-seed{seed}-pair{index:02d}.pt"
            partner_index = index + 1 if index % 2 == 0 else index - 1
            partner_path = run_dir / f"prefix-k{token_count}-seed{seed}-pair{partner_index:02d}.pt"
            payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
            partner = torch.load(partner_path, map_location="cpu", weights_only=True)
            if (payload["run_hash"] != run_hash or partner["run_hash"] != run_hash
                    or payload["train_id"] != train_example.example_id
                    or partner["train_id"] != selected_pairs[partner_index][0].example_id):
                raise RuntimeError("Checkpoint identity mismatch.")
            fit = payload["fit"]
            prefix = fit["prefix"]
            conditions = {"correct": prefix, "wrong": partner["fit"]["prefix"], "zero": torch.zeros_like(prefix)}
            predictions, scores, timings = {}, {}, {}
            for split, example in (("train", train_example), ("test", test_example)):
                for condition, candidate in conditions.items():
                    started = time.perf_counter()
                    prediction = generate_from_prefix(candidate, example.query)
                    label = f"{split}_{condition}"
                    predictions[label] = prediction
                    scores[label] = score_answer(prediction, example.answer)
                    timings[label] = time.perf_counter() - started
            memory = memory_for(train_example)
            result = {
                "index": index, "token_count": token_count, "seed": seed,
                "memory_type": memory.type, "postulate": memory.text,
                "train_question": train_example.query, "test_question": test_example.query,
                "gold_answer": test_example.answer, "train_prediction": predictions["train_correct"],
                "test_prediction": predictions["test_correct"],
                "train_correct": scores["train_correct"]["strict"], "test_correct": scores["test_correct"]["strict"],
                "predictions": predictions, "scores": scores, "generation_seconds": timings,
                "initial_loss": fit["losses"][0], "final_loss": fit["final_loss"],
                "losses": fit["losses"], "trace": fit["trace"], "eos_loss": fit["eos_loss"],
                "answer_token_ids": fit["answer_token_ids"], "answer_token_losses": fit["answer_token_losses"],
                "train_seconds": fit["train_seconds"], "peak_gpu_bytes": fit["peak_gpu_bytes"],
                "checkpoint_sha256": hashlib.sha256(checkpoint_path.read_bytes()).hexdigest(),
                "wrong_postulate": memory_for(selected_pairs[partner_index][0]).text,
            }
            results.append(result)
            atomic_json(result_path, results)
            result_keys.add(key)
            print(f"Saved k={token_count} seed={seed} pair={index + 1}: {scores['test_correct']}", flush=True)
            del payload, partner, fit, prefix, conditions
print("Completed fits/evaluations:", len(results), "Results:", result_path)


In [ ]:
# Persisted incrementally in the sweep; this also exports a final snapshot.
atomic_json(run_dir / "results.json", results)
print("Results and CPU prefix checkpoints are already on Drive:", run_dir)


In [ ]:
# No late copy is required: every completed unit is saved directly to Drive.
print("Manifest:", run_dir / "manifest.json")
print("Baselines:", run_dir / "baselines.json")


In [ ]:
from collections import defaultdict

print("Baseline accuracy (strict / approved alias):")
for condition in ("text_memory", "no_memory"):
    items = [item for item in baselines if item["condition"] == condition]
    if items:
        print(condition, len(items), {metric: sum(item["scores"][metric] for item in items) / len(items)
                                     for metric in ("strict", "alias")})

print("\nCapacity results by seed; each row uses 12 distinct postulates:")
for token_count in TOKEN_COUNTS:
    for seed in SEEDS:
        items = [item for item in results if item["token_count"] == token_count and item["seed"] == seed]
        if not items:
            continue
        metrics = {condition: {metric: sum(item["scores"][condition][metric] for item in items) / len(items)
                              for metric in ("strict", "alias")}
                   for condition in ("train_correct", "test_correct", "test_wrong", "test_zero")}
        print(f"k={token_count} seed={seed} n={len(items)}", metrics)

print("\nPer-type held-out strict accuracy (mean across seed rows):")
type_results = defaultdict(list)
for item in results:
    type_results[(item["token_count"], item["memory_type"])].append(item)
for (token_count, memory_type), items in sorted(type_results.items()):
    print(token_count, memory_type, sum(item["test_correct"] for item in items) / len(items))
print("Seed repetitions are not independent new postulates; use group-clustered intervals for final reporting.")


In [ ]:
for token_count in TOKEN_COUNTS:
    failures = [item for item in results if item["token_count"] == token_count and not item["test_correct"]]
    print(f"\nStrict failures at k={token_count}: {len(failures)} seed/example rows")
    for item in failures:
        print("Seed:", item["seed"], "Type:", item["memory_type"])
        print("Postulate:", item["postulate"])
        print("Test question:", item["test_question"])
        print("Gold:", repr(item["gold_answer"]))
        print("Prediction:", repr(item["test_prediction"]))
        print("Strict / alias:", item["scores"]["test_correct"])
        print("Answer losses:", item["answer_token_losses"], "EOS loss:", item["eos_loss"])
        print("Wrong-prefix prediction:", repr(item["predictions"]["test_wrong"]))
